From an Google Search 'Word2Vec' (https://www.google.com/search?q=word2vec&sca_esv=e755c4fcff4cb9a6&rlz=1C1ONGR_enUS1065US1065&sxsrf=APpeQnse_RvzfuT5h5oe8j2BE1mcGgX8LA%3A1785529232767&ei=kANtatmzLrmXruEP-OKzgQk&biw=2874.6865234375&bih=1066.833740234375&sclient=gws-wiz-serp&fbs=ABfTbFVyMZGZf1hfvX9uKjN_-G8c4u0nXx4bEIpwm1lnNH832VstEKsVDqPorK0Gahnm2no1YAFtlsByIZaJlK7yr6gIShz8_nfnRyCFKBFanfbilXpMs-cznwqr4eRh15jLYnTY1jneHErIL1s8ylJ677g0-Yzv9SeiVzusgosrLmIdC_Li_URL_fqqHo09-SPTQ8fS7ou1p27xw7ju_YWEy8MXRBbSOQ&aep=10&ntc=1&mstk=AUtExfDjGCbGLkqwvITjmZJS3lyGsQh-otF9QlZqdVr5gU5RMD2IR2k1L8GgtstNNOkW8V58BHVtijYOvkBDi7NBM5A_oMG-Zf0baGofEpssl-8tXTTBHYJ-zAyQf1SsQaucVv5i4H8Y9dLE6rxIA0H8BWHQDetvUAovqUuPvmgMDxNYH-jXqmnl36Mo1P1SpuyQKJJ34T4c3k2XTIGzOnTpSBNSxkUATZYU0fZjdRK4CXHm30zQHWnNHqBKrc5SRmytvIXI2UnV4_CWRQ&aioh=3&csuir=1&cs=0&sourceid=chrome&ccb=1&hl=en-US&atvm=1&mtid=NgRtau_2NaPrmLQPuaTfsAg&udm=50)



In [1]:
# Step 1: Install the Necessary Libraries

!pip install transformers datasets scikit-learn pandas


In [2]:
# Step 2: Load a Pre-trained Medical BERT Model:

import torch
from transformers import AutoTokenizer, AutoModel

# Check that PyTorch sees your GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the medical BERT tokenizer and model (BioBERT)
model_name = "dmis-lab/biobert-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

print("✅ Medical BERT model successfully loaded onto your GPU!")


Using device: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Medical BERT model successfully loaded onto your GPU!


In [3]:
# Step 4: Create the Sample Dataset in Colab:

import pandas as pd

# Create a sample clinical dataset with a built-in anomaly
data = {
    'encounter_id': [101, 102, 103, 104, 105],
    'specialty': ['Cardiology', 'Orthopedics', 'Pediatrics', 'Neurology', 'Gastroenterology'],
    'clinical_notes': [
        "Patient presents with intermittent chest pain and shortness of breath during exertion. EKG shows minor ST changes. Scheduled for a stress test.",
        "A 45-year-old male with a closed fracture of the right radius following a fall. Closed reduction performed and fiberglass cast applied.",
        "Routine 2-year well-child visit. Growth charts tracking at the 60th percentile. Immunizations updated. Normal developmental milestones met.",
        "Patient complains of acute, severe crushing chest pain radiating to the left jaw, accompanied by diaphoresis and nausea. Emergency EKG ordered.", # ANOMALY: Cardiology text labeled as Neurology
        "Subjective complaints of chronic abdominal bloating and epigastric burning after meals. Recommended upper endoscopy to rule out GERD or gastritis."
    ]
}

df = pd.DataFrame(data)
print(f"Dataset created successfully! Total records: {len(df)}")
print(df[['encounter_id', 'specialty']])


Dataset created successfully! Total records: 5
   encounter_id         specialty
0           101        Cardiology
1           102       Orthopedics
2           103        Pediatrics
3           104         Neurology
4           105  Gastroenterology


In [5]:
import torch
import numpy as np

# Ensure your model from the previous step is active
# model.eval() # This line should ideally be called once, outside the function or in an earlier setup cell

def get_bert_embedding(text):
    # Tokenize text and move tensors to the GPU
    inputs = tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Extract the 'Pooler Output' which represents the semantic meaning of the entire text block
    embeddings = outputs.pooler_output.cpu().numpy()
    return embeddings[0]

# Step 4.5  Create a unified text field for the model to read (inserted in later)

df['combined_text'] = "Specialty: " + df['specialty'] + " | Notes: " + df['clinical_notes']

# Function to get embeddings (using the function we defined in the previous step)
print("Generating contextual embeddings for combined text...")
embeddings_list = []
for text in df['combined_text']:
    # Get vector and squeeze to flatten from (1, 768) to (768,)
    vector = get_bert_embedding(text).squeeze()
    embeddings_list.append(vector)

# Convert list to a clean 2D NumPy array for machine learning
X = np.array(embeddings_list)
print(f"Matrix shape: {X.shape} (5 records, 768 mathematical features each)")

Generating contextual embeddings for combined text...
Matrix shape: (5, 768) (5 records, 768 mathematical features each)


In [8]:

print(X)
type(X)

[[ 0.06670038  0.08933568  0.99820006 ...  0.99937934  0.12188272
   0.9999998 ]
 [-0.11582005  0.14957272  0.9914662  ...  0.9963236   0.06964001
   0.9999991 ]
 [ 0.0055685   0.02964002  0.9973547  ...  0.99940205 -0.03130087
   0.99999976]
 [ 0.03829664  0.11181967  0.9912855  ...  0.9963171   0.0233459
   0.9999983 ]
 [-0.05041631  0.20762847  0.99801534 ...  0.99944085  0.01162135
   0.9999999 ]]


numpy.ndarray

In [9]:
# Step 5: Convert Text into Medical BERT Embeddings:

import torch
import numpy as np

# Ensure your model from the previous step is active
model.eval()

def get_bert_embedding(text):
    # Tokenize text and move tensors to the GPU
    inputs = tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Extract the 'Pooler Output' which represents the semantic meaning of the entire text block
    embeddings = outputs.pooler_output.cpu().numpy()
    return embeddings[0]

# Generate vectors for all clinical notes
print("Processing text through BioBERT on the GPU...")
df['vector'] = df['clinical_notes'].apply(get_bert_embedding)
print("✅ Text converted into dense math vectors successfully!")



Processing text through BioBERT on the GPU...
✅ Text converted into dense math vectors successfully!


In [10]:
# Step 6: Detect the Anomaly Using Isolation Forest:


from sklearn.ensemble import IsolationForest

# Initialize Isolation Forest
# contamination=0.20 tells the AI we expect roughly 20% of our data (1 out of 5) to be anomalous
# iso_forest = IsolationForest(contamination=0.20, random_state=42)

# Initialize the forest WITHOUT a hardcoded contamination percentage
iso_forest_raw = IsolationForest(contamination='auto', random_state=42)
iso_forest_raw.fit(X)

# Train the model on our BERT vectors and predict outliers
# Predictions return: 1 for normal data, -1 for anomalies
df['anomaly_score'] = iso_forest_raw.predict(X)

# Display the results
print("\n--- DETECTED ANOMALIES ---")
for index, row in df.iterrows():
    status = "⚠️ ANOMALY DETECTED" if row['anomaly_score'] == -1 else "✅ Normal Record"
    print(f"\nEncounter ID: {row['encounter_id']} | Status: {status}")
    print(f"Assigned Specialty: {row['specialty']}")
    print(f"Notes snippet: {row['clinical_notes'][:80]}...")

# Extract raw anomaly scores (lower/more negative = more anomalous)
# Note: sklearn offsets the score so that negative numbers are outliers
df['raw_anomaly_score'] = iso_forest_raw.score_samples(X)

# Sort records to show the most suspicious ones at the very top
df_sorted = df.sort_values(by='raw_anomaly_score')
print("\n--- SORTED RECORDS ---")
print(df_sorted[['encounter_id', 'specialty', 'raw_anomaly_score']])

NameError: name 'iso_forest' is not defined

In [ ]:
# Step 6: Detect the Anomaly Using Isolation Forest:


from sklearn.ensemble import IsolationForest

# Initialize Isolation Forest
# contamination=0.20 tells the AI we expect roughly 20% of our data (1 out of 5) to be anomalous
# iso_forest = IsolationForest(contamination=0.20, random_state=42)

# Initialize the forest WITHOUT a hardcoded contamination percentage
iso_forest_raw = IsolationForest(contamination='auto', random_state=42)
iso_forest_raw.fit(X)

# Train the model on our BERT vectors and predict outliers
# Predictions return: 1 for normal data, -1 for anomalies
df['anomaly_score'] = iso_forest_raw.predict(X)

# Display the results
print("\n--- DETECTED ANOMALIES ---")
for index, row in df.iterrows():
    status = "⚠️ ANOMALY DETECTED" if row['anomaly_score'] == -1 else "✅ Normal Record"
    print(f"\nEncounter ID: {row['encounter_id']} | Status: {status}")
    print(f"Assigned Specialty: {row['specialty']}")
    print(f"Notes snippet: {row['clinical_notes'][:80]}...")

# Extract raw anomaly scores (lower/more negative = more anomalous)
# Note: sklearn offsets the score so that negative numbers are outliers
df['raw_anomaly_score'] = iso_forest_raw.score_samples(X)

# Sort records to show the most suspicious ones at the very top
df_sorted = df.sort_values(by='raw_anomaly_score')
print("\n--- SORTED RECORDS ---")
print(df_sorted[['encounter_id', 'specialty', 'raw_anomaly_score']])

NameError: name 'iso_forest' is not defined

**Dynamically Setting the Threshold. **Once you have thousands of records, you cannot look at them one by one. You use statistical rules of thumb to isolate the threshold programmatically:


*   The Standard Deviation Rule (Extreme Outliers): Calculate the average score of your dataset. Flag any record that sits more than 2 or 3 standard deviations away from that average.
*   The Percentile Method for Auditing Budgets: If your quality assurance department only has the manpower to review 50 records a week, you simply sort your database by the lowest raw_anomaly_score and pull the Top 50 worst scores, regardless of percentage.








In [ ]:
# Step 7: Try This Statistical Thresholding Script

import numpy as np

scores = df['raw_anomaly_score'].values
mean_score = np.mean(scores)
std_score = np.std(scores)

# Define a statistical threshold (e.g., 1 standard deviation away for this tiny dataset)
# For large datasets, 2 or 3 standard deviations is standard
threshold = mean_score - (1.0 * std_score)

df['dynamic_anomaly_flag'] = df['raw_anomaly_score'].apply(lambda x: "⚠️ ANOMALY" if x < threshold else "✅ Normal")

print(f"Calculated Dataset Mean Score: {mean_score:.4f}")
print(f"Statistical Anomaly Cutoff Threshold: {threshold:.4f}\n")
print(df[['encounter_id', 'specialty', 'raw_anomaly_score', 'dynamic_anomaly_flag']])


In [ ]:
# Step 8: This is a correction script, to suggest to which specialty a record should be assigned to
# in this case, for illustration, it uses the known value of the anomalous case (4), but could
# be modified to pull case record numbers whose values were based on thresholds.

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Step 1: Isolate our known "good" clean records to build baseline profiles
# We exclude Encounter 4 because we already know it is an anomaly
clean_df = df[df['encounter_id'] != 104] # Changed from 4 to 104

# Create a dictionary to store the baseline vector for each specialty
specialty_baselines = {}
for spec in clean_df['specialty'].unique():
    # Get all vectors belonging to this specialty
    spec_vectors = np.array(clean_df[clean_df['specialty'] == spec]['vector'].tolist())
    # Calculate the average vector profile for this specialty
    specialty_baselines[spec] = np.mean(spec_vectors, axis=0).reshape(1, -1)

# Step 2: Grab the vector of our anomalous record (Encounter 104)
anomaly_vector = df[df['encounter_id'] == 104]['vector'].values[0].reshape(1, -1) # Changed from 4 to 104

# Step 3: Calculate similarity between the anomaly and every baseline profile
print("--- COGSINE SIMILARITY ANALYSIS FOR ENCOUNTER 104 ---") # Changed from 4 to 104
print(f"Current Mislabeled Specialty: {df[df['encounter_id'] == 104]['specialty'].values[0]}\n") # Changed from 4 to 104

highest_score = -1
recommended_specialty = None

for spec, baseline_vector in specialty_baselines.items():
    # Calculate cosine similarity (returns a 2D array, so we grab the scalar value)
    similarity_score = cosine_similarity(anomaly_vector, baseline_vector)[0][0]
    print(f"Similarity to {spec} baseline: {similarity_score:.4f}")

    if similarity_score > highest_score:
        highest_score = similarity_score
        recommended_specialty = spec

print("\n--- SYSTEM RECOMMENDATION ---")
print(f"💡 This record looks like an error. It should be re-assigned to: **{recommended_specialty}**")
print(f"Confidence score: {highest_score:.4f}")

Open this window, and type "Resume medical anomaly project,"

https://www.google.com/search?q=word2vec&sca_esv=e755c4fcff4cb9a6&rlz=1C1ONGR_enUS1065US1065&sxsrf=APpeQnse_RvzfuT5h5oe8j2BE1mcGgX8LA%3A1785529232767&ei=kANtatmzLrmXruEP-OKzgQk&biw=2874.6865234375&bih=1066.833740234375&sclient=gws-wiz-serp&fbs=ABfTbFVyMZGZf1hfvX9uKjN_-G8c4u0nXx4bEIpwm1lnNH832VstEKsVDqPorK0Gahnm2no1YAFtlsByIZaJlK7yr6gIShz8_nfnRyCFKBFanfbilXpMs-cznwqr4eRh15jLYnTY1jneHErIL1s8ylJ677g0-Yzv9SeiVzusgosrLmIdC_Li_URL_fqqHo09-SPTQ8fS7ou1p27xw7ju_YWEy8MXRBbSOQ&aep=10&ntc=1&mstk=AUtExfCORGHfCOoAuq7N4r7jnULryoQP9ma9pC4jQQXipsGY5RxDx0SoU3-_i6yEwlmdL02lpsDg5WHN79RU6d1S282tbqz_6-uzWKm5h9aoEYdfP06eb30U-uV41FDy0NOlRS5qIqTeaZ56PXp6UsOfDSXoXjG0GN1TmSqmO6nuaVsluOGtn9inSl004f0U0f-fpjbAjYYzyTf8daJsfBTjIgUu5T5CImeclcIlje7YHXWOl9rOozVfpK7t-elnPHzcIoA7Bkz3MEvI2A&aioh=3&csuir=1&cs=0&sourceid=chrome&ccb=1&hl=en-US&atvm=1&mtid=NgRtau_2NaPrmLQPuaTfsAg&udm=50